# CONFIG

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


%load_ext autoreload
%autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/BIOM/notebooks
/home/turbotowerlnx/Documents/Master/BIOM/app


In [2]:
import numpy as np
from tqdm import tqdm

from maikol_utils.print_utils import print_separator, print_warn, print_color

from src.config import Configuration    


CONFIG = Configuration(
    max_stages=5,
    objective_fpr=0.05
)

# seed all
np.random.seed(CONFIG.seed)

# GET ALL POSSIBLE HAAR

- We no dot create, for a single HAAR, both versions white-black / black-white, only one
- Since one is just the other $\times -1$, the weak classifiers will handle that by multiplying by $1$ or $-1$ as needed 

In [3]:
from typing import List
from src.model import Feature, Rectangle


def generate_all_features(win_w: int = 24, win_h: int = 24) -> List[Feature]:
    features = []
    
    for w in range(1, win_w + 1):
        for h in range(1, win_h + 1):
            for x in range(win_w - w + 1):
                for y in range(win_h - h + 1):
                    
                    # 2-rectangle horizontal (Left/Right)
                    if w % 2 == 0:
                        w_half = w // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w_half, h, -1.0),
                            Rectangle(x + w_half, y, w_half, h, 1.0)
                        ]))
                        
                    # 2-Rectangle vertical (Top/Bottom)
                    if h % 2 == 0:
                        h_half = h // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w, h_half, -1.0),
                            Rectangle(x, y + h_half, w, h_half, 1.0)
                        ]))
                        
                    # # 3-Rectangle horizontal (Left/Center/Right)
                    # if w % 3 == 0:
                    #     w_third = w // 3
                    #     features.append(Feature(
                    #         feature_id = len(features),
                    #         rectangles = [
                    #         Rectangle(x, y, w_third, h, -1.0),
                    #         Rectangle(x + w_third, y, w_third, h, 2.0), # Center weight compensates for 2 outside Rectangles
                    #         Rectangle(x + 2 * w_third, y, w_third, h, -1.0)
                    #     ]))
                        
                    # # 3-Rectangle vertical (Top/Center/Bottom)
                    # if h % 3 == 0:
                    #     h_third = h // 3
                    #     features.append(Feature(
                    #         feature_id = len(features),
                    #         rectangles = [
                    #         Rectangle(x, y, w, h_third, -1.0),
                    #         Rectangle(x, y + h_third, w, h_third, 2.0),
                    #         Rectangle(x, y + 2 * h_third, w, h_third, -1.0)
                    #     ]))
                        
                    # # 4-Rectangle (Checkerboard)
                    # if w % 2 == 0 and h % 2 == 0:
                    #     w_half, h_half = w // 2, h // 2
                    #     features.append(Feature(
                    #         feature_id = len(features),
                    #         rectangles = [
                    #         Rectangle(x, y, w_half, h_half, 1.0),
                    #         Rectangle(x + w_half, y, w_half, h_half, -1.0),
                    #         Rectangle(x, y + h_half, w_half, h_half, -1.0),
                    #         Rectangle(x + w_half, y + h_half, w_half, h_half, 1.0)
                    #     ]))
                        
    return features

In [4]:
all_features = generate_all_features(
    win_w = CONFIG.crop_size, 
    win_h = CONFIG.crop_size
)
len(all_features)

86400

# LOAD DATA

In [5]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from maikol_utils.file_utils import load_json, list_dir_files

# =============== FACES DATASET =============== 
all_faces, n = list_dir_files(CONFIG.faces_path, recursive=True)
print(f" - Found {n} files in {CONFIG.faces_path}")


# =============== NO-FACES DATASET =============== 
to_keep_labels = load_json(CONFIG.dataset_classes_path)

# Download dataset without faces
bg_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=to_keep_labels,
    max_samples=5000,
    dataset_dir=CONFIG.no_faces_path
)
bg_dataset = bg_dataset.filter_labels("ground_truth", F("label").is_in(to_keep_labels))


/home/turbotowerlnx/Documents/Master/BIOM/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 - Found 136965 files in ../data/ViolaJones/face_images
Loading output from ../data/dataset_classes.json...
Necessary images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'open-images-v7-train-5000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


### PRECOMPUTE ALL FEATURES FOR FACES DATASET

In [6]:
all_faces = np.random.choice(all_faces, size=1500, replace=False)
print(f"Using {len(all_faces)} files in {CONFIG.faces_path}")

Using 1500 files in ../data/ViolaJones/face_images


In [7]:
import numpy as np
from functools import lru_cache
from src.data import get_integral_image, get_integral_squared_image, get_std_from_integral_images, get_integral_sum, get_image_crops_from_list

# ── Precompute feature geometry ONCE at module load ──────────────────────────
# Shape: (total_rectangles,) flat arrays — avoids all Python loops at runtime

def _precompute_feature_tensors(features):
    r1s, c1s, r2s, c2s, ws, fidx = [], [], [], [], [], []
    for i, feat in enumerate(features):
        for rec in feat.rectangles:
            r1s.append(rec.y)
            c1s.append(rec.x)
            r2s.append(rec.y + rec.height - 1)
            c2s.append(rec.x + rec.width - 1)
            ws.append(rec.weight)
            fidx.append(i)
    return (
        np.array(r1s,  dtype=np.int32),
        np.array(c1s,  dtype=np.int32),
        np.array(r2s,  dtype=np.int32),
        np.array(c2s,  dtype=np.int32),
        np.array(ws,   dtype=np.float16),
        np.array(fidx, dtype=np.int32),
    )

# Call once — store as module-level constants
_R1, _C1, _R2, _C2, _W, _FIDX = _precompute_feature_tensors(all_features)
_N_FEATURES = len(all_features)


# ── Vectorized: ALL features for ONE integral image ───────────────────────────

def _compute_features_vectorized(integral_img: np.ndarray) -> np.ndarray:
    """Compute all Haar features for one integral image — zero Python loops."""
    II = integral_img

    # Integral sum for every rectangle in one shot (vectorized corner lookup)
    vals = II[_R2, _C2].copy()
    mask_r = _R1 > 0
    mask_c = _C1 > 0
    vals[mask_r]           -= II[_R1[mask_r] - 1, _C2[mask_r]]
    vals[mask_c]           -= II[_R2[mask_c], _C1[mask_c] - 1]
    vals[mask_r & mask_c]  += II[_R1[mask_r & mask_c] - 1, _C1[mask_r & mask_c] - 1]

    # Weighted sum, then accumulate into per-feature bins
    return np.bincount(_FIDX, weights=vals * _W, minlength=_N_FEATURES).astype(np.float16)


# ── Vectorized: ALL crops from one image in one batch ─────────────────────────

def extract_features_batch(imgs: list[np.ndarray]) -> np.ndarray:
    """
    Extract features for a list of crop images.
    Returns shape (n_crops, n_features).
    """
    results = np.empty((len(imgs), _N_FEATURES), dtype=np.float16)
    for i, img in enumerate(imgs):
        integral_img   = get_integral_image(img)
        integral_img_2 = get_integral_squared_image(img)

        H, W = img.shape
        _, std_dev = get_std_from_integral_images(
            integral_img, integral_img_2,
            np.array([[0]]), np.array([[H - 1]]),
            np.array([[0]]), np.array([[W - 1]]),
        )
        std_dev = float(std_dev[0, 0]) or 1.0

        results[i] = _compute_features_vectorized(integral_img) / std_dev

    return results


# ── process_image: crops extracted then batch-featurized ──────────────────────

def process_image(classifier, filepath):
    fps = classifier.predict_no_merge(img_path=filepath)
    if not fps:
        return []
    crops = get_image_crops_from_list(fps, img_path=filepath)
    if not crops:
        return []

    imgs = [crop["img"] for crop in crops]
    batch = extract_features_batch(imgs)   # (n_crops, n_features) — one numpy call
    return list(batch)

In [8]:
import os
import cv2
import multiprocessing as mp
from src.data import get_integral_image, get_integral_squared_image, get_std_from_integral_images, get_integral_sum
from src.model import compute_feature

# def compute_features_batch(integral_img, all_features):
#     """Compute all features at once using vectorized numpy ops instead of per-feature Python loop."""
#     results = np.zeros(len(all_features), dtype=np.float16)
#     for i, feature in enumerate(all_features):
#         feature_sum = 0.0
#         for rec in feature.rectangles:
#             feature_sum += get_integral_sum(
#                 integral_img,
#                 rec.y, rec.x,
#                 rec.y + rec.height - 1, rec.x + rec.width - 1
#             ) * rec.weight
#         results[i] = feature_sum
#     return results

# def extract_features(img_path: str = None, img: np.ndarray = None) -> np.ndarray:
#     if img is None:
#         img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

#     integral_img = get_integral_image(img)
#     integral_img_2 = get_integral_squared_image(img)

#     H, W = img.shape
#     r1, r2 = np.array([[0]]), np.array([[H - 1]])
#     c1, c2 = np.array([[0]]), np.array([[W - 1]])

#     _, std_dev = get_std_from_integral_images(integral_img, integral_img_2, r1, r2, c1, c2)
#     std_dev = float(std_dev[0, 0]) or 1.0
#     # print(" - Extracting features...")
#     return compute_features_batch(integral_img, all_features) / std_dev
def extract_features(img_path: str = None, img: np.ndarray = None) -> np.ndarray:
    if img is None:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    integral_img = get_integral_image(img)
    integral_img_2 = get_integral_squared_image(img)

    # Calculate std
    H, W = img.shape
    r1 = np.array([[0]])
    r2 = np.array([[H - 1]])
    c1 = np.array([[0]])
    c2 = np.array([[W - 1]])

    _, std_dev = get_std_from_integral_images(integral_img, integral_img_2, r1, r2, c1, c2)
    std_dev = std_dev[0, 0]

    if std_dev <= 0:
        std_dev = 1.0

    res = _compute_features_vectorized(integral_img) / std_dev
    del integral_img, integral_img_2
    return res

def compute_features_dataset(images_paths):
    n_workers = int(max(1, (os.cpu_count() or 1) - 1) * 4 / 5)
    chunksize = 8

    with mp.get_context("fork").Pool(processes=n_workers) as pool:
        results_faces = list(
            tqdm(
                pool.imap(extract_features, images_paths, chunksize=chunksize),
                total=len(images_paths),
                desc=f"Extracting face features ({n_workers} workers)",
            )
        )

    features = np.stack(results_faces, axis=0).astype(np.float16, copy=False)
    return features


if os.path.exists(CONFIG.faces_np_path):
    print(f" - Loading precomputed face features from {CONFIG.faces_np_path}...")
    X_train_faces = np.load(CONFIG.faces_np_path)
    print(f" - Loaded face features for {X_train_faces.shape[0]} images.")
    print(f" - X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")
else:
    X_train_faces = compute_features_dataset(all_faces)
    np.save(CONFIG.faces_np_path, X_train_faces)
    print(f" - Computed face features for {X_train_faces.shape[0]} images.")
    print(f" - X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")


 - Loading precomputed face features from ../data/ViolaJones/faces.npy...
 - Loaded face features for 1500 images.
 - X_train_faces dtype=float32, shape=(1500, 86400)


# VIOLA JONES STAGES

### Train AdaBoost
- Train with stumps, using at most x features per stage
- Then check at which point of the trained, we mee the True positive and False positive rate
- Remove unused stages 
- Return remaining tree and threshold for that stage

In [9]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np

def train_stage_early_stopping(X_train, y_train, max_features=200, target_tpr=0.995, target_fpr=0.50):
    """
    train_stage_for_tpr trains an AdaBoost classifier and determines a custom threshold to achieve the target true positive rate (TPR) on the training data.

    X_train shape: (num_images, num_features) - precalculated feature values
    y_train shape: (num_images,) - 1 for face, 0 for background

    returns:
    - clf: the trained AdaBoost classifier
    - custom_threshold: the decision function threshold to achieve the target TPR on the training data
    """
    print(' - Fitting AdaBoost with early stopping...')
    clf = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=max_features, # set to a cap for bc no need to check all
    )
    clf.fit(X_train, y_train)

    X_faces = X_train[y_train == 1]
    X_bg = X_train[y_train == 0]

    print(' - Refining threshold and selecting features...')
    # .staged_decision_function evaluates the ensemble using 1 feature, then 2 features, etc.
    for i, (face_scores, bg_scores) in enumerate(zip(
        clf.staged_decision_function(X_faces),
        clf.staged_decision_function(X_bg)
    )):
        
        # Force the threshold to meet the 99.5% TPR target
        face_scores_sorted = np.sort(face_scores)
        drop_count = int(len(face_scores) * (1.0 - target_tpr))
        custom_threshold = face_scores_sorted[drop_count]

        # Check the False Positive Rate using this forced threshold
        false_positives = np.sum(bg_scores >= custom_threshold)
        fpr = false_positives / len(X_bg)

        print(f"   - Features: {i+1} | FPR: {fpr:.3f}")

        # 3. Stop early if we hit the FPR target!
        if fpr <= target_fpr:
            print(f" - Stage criteria met! Stopping at {i+1} features.")
            
            # Truncate the sklearn classifier to drop the unused extra features
            clf.estimators_ = clf.estimators_[:i+1]
            clf.estimator_weights_ = clf.estimator_weights_[:i+1]
            clf.estimator_errors_ = clf.estimator_errors_[:i+1]
            clf.classes_ = np.array([0, 1])
            
            return clf, custom_threshold

    print_warn("Max features reached without hitting FPR target.")
    return clf, custom_threshold

# To predict later: 
# passes_stage = clf.decision_function(X_test) >= custom_threshold

### Generate hard mining samples
- Load random images without faces and get crops that PASS all the current stages


In [10]:
import dataclasses

from src.model import build_haar_cascade_from_stages, CascadeClassifier
from src.data import get_image_crops_from_list
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
from tqdm import tqdm
import threading
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

def balance_non_face_samples(stages, num_samples, bg_samples, max_iterations=10000, n_workers=8):
    cascade = build_haar_cascade_from_stages(
        stages_output=stages,
        all_features=all_features,
        width=CONFIG.crop_size,
        height=CONFIG.crop_size,
        cascade_type="trained_adaboost_stages",
        feature_type="HAAR",
    )
    classifier = CascadeClassifier(dataclasses.replace(CONFIG, stride=4), cascade)

    stop_event = threading.Event()

    def process_image(filepath, max_crops):
        if stop_event.is_set():
            return []
        fps = classifier.predict_no_merge(img_path=filepath)
        if not fps:
            return []
        crops = get_image_crops_from_list(fps, img_path=filepath)
        if not crops:
            return []
        # Cap how many crops we extract to avoid overshoot
        crops = crops[:max_crops]
        return list(extract_features_batch([c["img"] for c in crops]))

    sample_size = min(max_iterations, len(bg_samples))
    filepaths = np.random.choice(bg_samples, size=sample_size, replace=False)

    print(f" - Generating {num_samples} hard negative samples ({n_workers} workers)...")
    all_hard_bg = []
    images_processed = 0

    with tqdm(total=num_samples, desc="Hard negatives", unit="crops") as pbar:
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            # Submit lazily — only keep a small buffer in flight at a time
            fp_iter = iter(filepaths)
            buffer_size = n_workers * 2
            futures = {}

            def submit_next():
                fp = next(fp_iter, None)
                if fp is None or stop_event.is_set():
                    return
                remaining = num_samples - len(all_hard_bg)
                f = executor.submit(process_image, fp, remaining)
                futures[f] = fp

            # Fill initial buffer
            for _ in range(buffer_size):
                submit_next()

            while futures:
                done, _ = wait(futures, return_when=FIRST_COMPLETED)
                for future in done:
                    futures.pop(future)
                    crops = future.result()
                    images_processed += 1

                    if crops:
                        all_hard_bg.extend(crops)
                        pbar.update(len(crops))

                    pbar.set_postfix(imgs=images_processed, crops=len(all_hard_bg))

                    if len(all_hard_bg) >= num_samples:
                        stop_event.set()
                        break

                    submit_next()  # refill buffer one at a time

                if stop_event.is_set():
                    break

    if len(all_hard_bg) < num_samples:
        print_warn(
            f"Only found {len(all_hard_bg)} crops (requested {num_samples}) "
            f"after processing {images_processed} images."
        )
    else:
        print(f" - Collected {len(all_hard_bg)} crops from {images_processed} images (target was {num_samples})")

    return np.array(all_hard_bg, dtype=np.float32)

### Generate all cascade stages
2 stopping criteria 
- reaching FPR of $10 e -6$. This is computed not as in the paper (they estimated it from each stage $fpr_i$), but rather from the number of windows that had to be scanned to create the new 'X_train_bg'.
- Hand number (say 50)


In [ ]:
def generate_all_stages(X_train_faces, bg_samples, max_stages=10, target_fpr=0.005):
    stages = []
    prev_n_faces = len(X_train_faces) 
    n_bg_pre = len(X_train_faces) 
    n_features = X_train_faces.shape[1]
    prev_fp = np.empty((0, n_features), dtype=np.float32) # Start with no hard negatives
    fpr_macro = 1.0

    for stage_num in range(max_stages):
        print_separator(f"Training stage {stage_num + 1}/{max_stages}", sep_type="LONG")
        print_separator("Generating hard negative samples")

        X_train_bg = balance_non_face_samples(
            stages, 
            num_samples=prev_n_faces - len(prev_fp), # Only add enough new bg samples to maintain balance with faces
            bg_samples=bg_samples
        )

        X_train = np.vstack((X_train_faces, X_train_bg, prev_fp))
        y_train = np.hstack((np.ones(len(X_train_faces)), np.zeros(len(X_train_bg)), np.zeros(len(prev_fp))))
        del X_train_bg 

        print_separator("Training")
        clf, threshold = train_stage_early_stopping(X_train, y_train)
        stages.append((clf, threshold))
        
        print_separator("Filtering: Hard Negative mining")
        decision_scores = clf.decision_function(X_train)
        passes_stage = decision_scores >= threshold
        X_train = X_train[passes_stage]
        y_train = y_train[passes_stage]
        
        prev_fp = X_train[y_train == 0]

        n_faces = np.sum(y_train == 1)
        n_bg = len(prev_fp)
        # macro FPR: Fi = Fi-1 * fpr_micro with F0 = 1.0  
        fpr_micro = n_bg / n_bg_pre
        fpr_macro *= fpr_micro

        print(f" - Stage {stage_num + 1} used {len(clf.estimators_)} features.")
        print(f" - After stage {stage_num + 1}, {len(X_train)} samples remain for training.")
        print(f"   - {n_faces} faces")
        print(f"   - {n_bg} non-faces")
        print(f"   - {fpr_micro:.4f} micro false positive rate")
        print(f"   - {fpr_macro:.4f} macro false positive rate")
        
        n_bg_pre = n_bg
        prev_n_faces = n_faces
        # Stop conditions
        if n_bg == 0:
            print_color("No more negative samples left. Stopping training.", color="green")
            break
        
        if fpr_macro <= target_fpr:
            print_color(f"Reached target FPR of {target_fpr:.4f}. Stopping training.", color="green")
            break

    return stages, fpr_macro

stages, fpr_macro = generate_all_stages(
    X_train_faces=X_train_faces,
    bg_samples=[sample.filepath for sample in bg_dataset],
    max_stages=CONFIG.max_stages,
    target_fpr=CONFIG.objective_fpr
)

________________________________________________________________________________________________________________________________
                                                       Training stage 1/5                                                       

________________________________________________________________
                Generating hard negative samples                

 - Generating 1500 hard negative samples (8 workers)...


Hard negatives: 100%|██████████| 1500/1500 [00:16<00:00, 90.30crops/s, crops=1500, imgs=1] 


 - Collected 1500 crops from 1 images (target was 1500)


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 86400 and the array at index 2 has size 0

# FINAL STAGES ClASSIFIER

### Save the trained cascade to XML

In [ ]:
from src.model import CascadeSerializer

haar_cascade = build_haar_cascade_from_stages(
    stages_output=stages,
    all_features=all_features,
    width=CONFIG.crop_size,
    height=CONFIG.crop_size,
    cascade_type="trained_adaboost_stages",
    feature_type="HAAR",
)

print_separator(f"FINAL CASCADE", sep_type="LONG")
print(f" - Stages: {len(haar_cascade.stages)}")
print(f" - Features used: {len(haar_cascade.features)}")
print(f" - Window size: {haar_cascade.width}x{haar_cascade.height}")
print(haar_cascade)


cascade_path = os.path.join(CONFIG.computed_haar_cascades, f"stages_vj-{fpr_macro:.4f}.xml")
CascadeSerializer.save(haar_cascade, cascade_path)

print(f" - File: {cascade_path}")

In [ ]:
loaded_cascade = CascadeSerializer.load(cascade_path)

---

In [ ]:
import os
import importlib
import cv2
import numpy as np
import pandas as pd

# Ensure notebook can import project modules when run standalone
if not os.getcwd().endswith("app"):
    os.chdir("../app")

from src.config import Configuration
from src.model import build_haar_cascade_from_stages
from src.model.haar_cascade_parser import load_cascade
from maikol_utils.file_utils import list_dir_files

import src.model.cascade_clasifier as cascade_clasifier_module
importlib.reload(cascade_clasifier_module)
CascadeClassifier = cascade_clasifier_module.CascadeClassifier

CONFIG = Configuration()

# Generate all possible Haar features (same family used during training)
from typing import List
from src.model import Feature, Rectangle

def generate_all_features(win_w: int = 24, win_h: int = 24) -> List[Feature]:
    features = []
    for w in range(1, win_w + 1):
        for h in range(1, win_h + 1):
            for x in range(win_w - w + 1):
                for y in range(win_h - h + 1):
                    if w % 2 == 0:
                        w_half = w // 2
                        features.append(Feature(
                            feature_id=len(features),
                            rectangles=[
                                Rectangle(x, y, w_half, h, -1.0),
                                Rectangle(x + w_half, y, w_half, h, 1.0),
                            ],
                        ))
                    if h % 2 == 0:
                        h_half = h // 2
                        features.append(Feature(
                            feature_id=len(features),
                            rectangles=[
                                Rectangle(x, y, w, h_half, -1.0),
                                Rectangle(x, y + h_half, w, h_half, 1.0),
                            ],
                        ))
    return features

all_features = generate_all_features(CONFIG.crop_size, CONFIG.crop_size)

# ---- Collect a few non-face image paths (with fallbacks) ----
search_roots = [
    CONFIG.no_faces_path,
    os.path.join(CONFIG.DATA_PATH, "others", "NaturalImages", "all"),
]

non_face_paths = []
source_root = None
for root in search_roots:
    paths, _ = list_dir_files(root, recursive=True)
    image_paths = [p for p in paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]
    if image_paths:
        non_face_paths = image_paths[:3]
        source_root = root
        break

print(f"Testing with {len(non_face_paths)} non-face images")
print(f"Source folder: {source_root}")
print(f"predict_no_merge available: {hasattr(CascadeClassifier, 'predict_no_merge')}")

# ---- Build classifier with EMPTY stages ----
empty_cascade = build_haar_cascade_from_stages(
    stages_output=[],
    all_features=all_features,
    width=CONFIG.crop_size,
    height=CONFIG.crop_size,
    cascade_type="trained_adaboost_stages",
    feature_type="HAAR",
)
empty_classifier = CascadeClassifier(CONFIG, empty_cascade)

# ---- Build classifier from OpenCV frontal-face cascade XML ----
haar_base = getattr(CONFIG, "haar_cascades", None) or CONFIG.cv_haar_cascades
cascade_path = os.path.join(haar_base, "haarcascade_frontalface_default.xml")
xml_cascade = load_cascade(cascade_path)
xml_classifier = CascadeClassifier(CONFIG, xml_cascade)

def run_test(classifier, image_paths, max_side=128):
    rows = []
    for p in image_paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is None:
            rows.append({
                "image": os.path.basename(p),
                "shape": None,
                "candidates": None,
                "raw_accepted": None,
                "grouped_detections": None,
                "error": "could not read image",
            })
            continue

        h, w = img.shape[:2]
        scale = min(1.0, max_side / float(max(h, w)))
        if scale < 1.0:
            img_small = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
        else:
            img_small = img

        raw_faces, n_candidates = classifier.predict_no_merge(
            img=img_small,
            return_candidate_count=True,
        )
        grouped_faces, _ = classifier.predict(img=img_small, return_candidate_count=True)

        rows.append({
            "image": os.path.basename(p),
            "shape": tuple(img_small.shape),
            "candidates": int(n_candidates),
            "raw_accepted": int(len(raw_faces)),
            "grouped_detections": int(len(grouped_faces)),
            "acceptance_ratio_raw": float(len(raw_faces) / n_candidates) if n_candidates > 0 else np.nan,
            "first_raw": raw_faces[0] if len(raw_faces) > 0 else None,
            "first_grouped": grouped_faces[0] if len(grouped_faces) > 0 else None,
        })
    return pd.DataFrame(rows)

print("\n=== EMPTY STAGES ===")
df_empty = run_test(empty_classifier, non_face_paths)
display(df_empty)

print("\n=== OPENCV FRONTALFACE XML ===")
df_xml = run_test(xml_classifier, non_face_paths)
display(df_xml)

print("\nCascade path used:")
print(cascade_path)

Testing with 3 non-face images
Source folder: ../data/others/NaturalImages/all
predict_no_merge available: True
Loading Haar cascade from: ../models/haar_cascades/haarcascade_frontalface_default.xml

=== EMPTY STAGES ===


,image,shape,candidates,raw_accepted,grouped_detections,acceptance_ratio_raw,first_raw,first_grouped
0,airplane_0000.jpg,"(44, 128)",252,252,1,1.0,"{'x': 0, 'y': 0, 'w': 24, 'h': 24}","{'x': 50, 'y': 8, 'w': 27, 'h': 27}"
1,airplane_0001.jpg,"(50, 128)",345,345,1,1.0,"{'x': 0, 'y': 0, 'w': 24, 'h': 24}","{'x': 49, 'y': 10, 'w': 28, 'h': 28}"
2,airplane_0002.jpg,"(41, 128)",210,210,1,1.0,"{'x': 0, 'y': 0, 'w': 24, 'h': 24}","{'x': 50, 'y': 7, 'w': 27, 'h': 27}"



=== OPENCV FRONTALFACE XML ===


,image,shape,candidates,raw_accepted,grouped_detections,acceptance_ratio_raw,first_raw,first_grouped
0,airplane_0000.jpg,"(44, 128)",252,0,0,0.0,None,None
1,airplane_0001.jpg,"(50, 128)",345,0,0,0.0,None,None
2,airplane_0002.jpg,"(41, 128)",210,0,0,0.0,None,None



Cascade path used:
../models/haar_cascades/haarcascade_frontalface_default.xml
